<a href="https://colab.research.google.com/github/elvingup/projeto_deteccao_python_data_analytics_20260821/blob/main/projeto_deteccao_python_data_analytics_20260821.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DETECÇÃO DE FRAUDES EM TRANSAÇÕES BANCÁRIAS DO DATASET CREDITCARD

In [ ]:
import pandas as pd

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(url)

df.head()

In [ ]:
# Determinar a quantidade de itens de cada valor correspondentes à feature "Class"

pd["Class"].value_counts(normalize=True)

In [ ]:
# FEATURING ENGINEERING: transformação algorítmica da feature "Amount"

import numpy as np

# criar a feature "Amount_log" referenciado-se na feature "Amount" visando calibrar a ESCALA DE DADOS visto que valores pequenos poderiam dificultar a IDENTIFICAÇÃO DE PADRÕES que podem estar presentes no dataset
df["Amount_log"] = np.log1p(df["Amount"])


In [ ]:
# FEATURING ENGINEERING: usar objeto StandardScaler para calcular a MÉDIA e o DESVIO PADRÃO da feature "Amount"

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# criar a feature "Amount_scaled" para ajustar a ESCALA DOS DADOS de modo a evitar o viés de alguma coluna com números muito maiores que o restante das colunas. Faz isso ao calcular a MÉDIA e o DESVIO PADRÃO da feature "Amount"
df["Amount_scaled"] = scaler.fit_transform(df["Amount"])

In [ ]:
# FEATURING ENGINEERING: treinar o modelo

from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)


In [ ]:
# Avaliação dos Modelos empregando Logistic Regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [ ]:
# Impressão do Relatório

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
# Curva ROC para conferir se o modelo não é aleatório

from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib as plt

y_probs = model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_probs)

plt.plot(fpr, tpr)
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# Se o modelo for preciso, o valor de AUC deve ser próximo de 1.00
print("AUC: ", roc_auc_score(y_test, y_probs))

In [ ]:
# Precision Recall Curve para comparar a "Quantidade de Fraudes Detectadas" contra a "Quantidade de Fraudes que são fraudes mesmo"

from sklearn.metrics import precision_recall_curve

precision, recall = precision_recall_curve(y_test, y_probs)

plt.plot(recall, precision)
plt.title("Precision Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

In [ ]:
# BALANCEAMENTO DE DADOS: Undersampling para reduzir a classe majoritária

fraudes = df[df["Class"] == 1]
normais = df[df["Class"] == 0].sample(len(fraudes), random_state = 42)

df_under = pd.concat([fraudes, normais])

In [ ]:
# BALANCEAMENTO DE DADOS: Oversampling para aumentar a classe minoritária

from imblearn.over_sampling import SMOTE

smote = SMOTE()

X_res, y_res = smote.resample(X, y)

In [ ]:
#  BALANCEAMENTO DE DADOS: Floresta de Árvores de Decisão

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight="balanced",
    n_jobs=-1
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))



In [ ]:
# PIPELINE

from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)


In [ ]:
# AJUSTE DE LIMIAR

threshold = 0.3

y_pred_custom = (y_probs > threshold).astype(int)

print(classification_report(y_test, y_pred_custom))


In [ ]:
# XGBoost - uso de boosting (sequencia de treinamentos de modelos simples que convergem para corrigir os treinamentos anteriores): algoritmo que é muito apreciado no Mercado

from xgboost import XGBClassifier

xgb = XGBoost(
    scale_pos_weight=10,
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)



In [ ]:
# Exibição do XGBoost

print(classification_report(y_test, y_pred_xgb))


In [ ]:
# Exibição da importância das variáveis

import matplotlib.pyplot as plt

importancias = xgb.feature_importances_

plt.bar(range(len(importancias)), importancias)
plt.title("Importância das Variáveis")
plt.show()


In [ ]:
# AJUSTE DE HIPERPARÂMETROS: Melhorar o Modelo ao testar várias combinações

# GridSearchCV testa várias combinações de parâmetros visando encontrar a melhor dentre elas.
from sklearn.model_selection import GridSearchCV

# Definição das combinações a serem testadas
param_grid = {
    "max_depth" = [3,5],
    "n_estimators" = [50,100]
}

# Treinamento de vários modelos (um modelo respectivamente a um parâmetro) e determinação do recall como critério de avaliação de bom desempenho
grid = GridSearchCV(
    XGBClassifier(eval_metric="logloss"),
    param_grid,
    scoring = "recall",
    cv = 3
)

grid.fit(X_train, y_train)

# Exibição do resultado da avaliação
print("Melhor modelo: ", grid.best_params_)

In [ ]:
# EXPLICABILIDADE (SHAP): mostra como cada feature contriubui para a decisão do modelo

# Invocação da funcionalidade SHAP
explainer = shap.Explainer(xgb)

# Seleção dos valores
shap_values = explainer(X_test[:100])

# Exibição dos valores
shap.plots.bar(shap_values)

### Métricas puras de Machine Learning como *Recall* e *Precision* mostram apenas o desempenho estático do algoritmo. Ao traduzir essas métricas para moeda corrente, vamos perceber que o modelo reduz o prejuízo operacional com fraudes: mesmo considerando o custo operacional e atrito de alarmes falsos.

In [ ]:
# Avaliação de Efeito Financeiro e Valor de Negócio

import seaborn as sns
from sklearn.metrics import confusion_matrix

def analisar_impacto_financeiro(y_true, y_pred, valores_transacoes, custo_fp=20.0):
    """
    Calcula o impacto financeiro do modelo de detecção de fraude.

    Parameters:
    - y_true: Rótulos reais (0 = Legítima, 1 = Fraude)
    - y_pred: Predições do modelo (0 ou 1)
    - valores_transacoes: Séries com os valores financeiros das transações (Amount)
    - custo_fp: Custo estimado de um alarme falso (Falso Positivo)
    """
    df_resultado = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
        'amount': valores_transacoes
    })

    # Matriz de Confusão
    tn, fp, fn, tp = confusion_matrix(df_resultado['y_true'], df_resultado['y_pred']).ravel()

    # Cálculo dos custos
    custo_total_fp = fp * custo_fp

    # Para o FN (Fraude não detectada), somamos o valor real das transações fraudulentas que passaram
    custo_total_fn = df_resultado[(df_resultado['y_true'] == 1) & (df_resultado['y_pred'] == 0)]['amount'].sum()

    # Custo total com o modelo operando
    custo_com_modelo = custo_total_fp + custo_total_fn

    # Custo baseline (Cenário sem modelo: aprovar todas as transações -> todas as fraudes viram FN)
    custo_sem_modelo = df_resultado[df_resultado['y_true'] == 1]['amount'].sum()

    # Economia gerada
    economia = custo_sem_modelo - custo_com_modelo
    roi_percentual = (economia / custo_sem_modelo) * 100 if custo_sem_modelo > 0 else 0

    # Exibição dos resultados estruturados
    print("="*60)
    print("           RELATÓRIO DE IMPACTO FINANCEIRO DO MODELO          ")
    print("="*60)
    print(f"• Total de Transações Avaliadas : {len(df_resultado):,}")
    print(f"• Total de Fraudes Reais        : {tp + fn:,} (R$ {custo_sem_modelo:,.2f})")
    print("-" * 60)
    print(f" [TP] Fraudes Evitadas          : {tp:,}")
    print(f" [FP] Alarmes Falsos            : {fp:,}  --> Custo Op.: R$ {custo_total_fp:,.2f}")
    print(f" [FN] Fraudes Não Detectadas    : {fn:,}  --> Prejuízo : R$ {custo_total_fn:,.2f}")
    print(f" [TN] Transações Legítimas OK   : {tn:,}")
    print("-" * 60)
    print(f" CUSTO TOTAL SEM MODELO        : R$ {custo_sem_modelo:,.2f}")
    print(f" CUSTO TOTAL COM MODELO        : R$ {custo_com_modelo:,.2f}")
    print(f" ECONOMIA LÍQUIDA GERADA       : R$ {economia:,.2f} ({roi_percentual:.2f}% de redução de perdas)")
    print("="*60)

    return {
        'custo_sem_modelo': custo_sem_modelo,
        'custo_com_modelo': custo_com_modelo,
        'economia': economia,
        'custo_fp': custo_total_fp,
        'custo_fn': custo_total_fn
    }

# --- EXEMPLO DE USO ---
y_pred_xgb = xgb_model.predict(X_test)
resultado_fin = analisar_impacto_financeiro(
    y_true=y_test,
    y_pred=y_pred_xgb,
    valores_transacoes=X_test['Amount'], # Ou a coluna referente ao valor no seu dataset de teste
    custo_fp=20.0
)

In [ ]:
# Otimização do Limiar de Decisão (Threshold Tuning)

def otimizar_threshold_financeiro(model, X_test, y_test, valores_transacoes, custo_fp=20.0):
    probas = model.predict_proba(X_test)[:, 1]
    thresholds = np.linspace(0.01, 0.99, 100)
    custos = []

    for t in thresholds:
        y_pred = (probas >= t).astype(int)

        # Falsos positivos
        fp = np.sum((y_pred == 1) & (y_test == 0))
        custo_fp_total = fp * custo_fp

        # Falsos negativos (soma dos valores)
        fn_mask = (y_pred == 0) & (y_test == 1)
        custo_fn_total = valores_transacoes[fn_mask].sum()

        custos.append(custo_fp_total + custo_fn_total)

    idx_melhor = np.argmin(custos)
    melhor_threshold = thresholds[idx_melhor]
    menor_custo = custos[idx_melhor]

    plt.figure(figsize=(10, 5))
    plt.plot(thresholds, custos, color='#1f77b4', linewidth=2, label='Custo Total (R$)')
    plt.axvline(melhor_threshold, color='red', linestyle='--', label=f'Threshold Ótimo: {melhor_threshold:.2f}')
    plt.title('Curva de Custo Financeiro por Limiar de Decisão')
    plt.xlabel('Threshold de Probabilidade')
    plt.ylabel('Prejuízo Total (R$)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f"Limiar ideal de corte financeiro: {melhor_threshold:.2f} com prejuízo mínimo de R$ {menor_custo:,.2f}")

# Executar otimização
otimizar_threshold_financeiro(xgb_model, X_test, y_test, X_test['Amount'])